# Coffee17 preprocessing — Kaggle validation decision

Tidak training dan tidak membuka outer test. Tambahkan **output notebook W0** sebagai Kaggle Input; output W0 harus sudah membawa hasil R0→C0→F0→W0 secara berantai.


In [ ]:
CODE_COMMIT='7de2abb46efd5e71dbab508e2ae4b4e61102a7aa'
import hashlib, importlib, json, os, shutil, subprocess, sys
from pathlib import Path
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
PROJECT=WORK/'coffee17-preprocessing-project'
REPO=WORK/'coffee-bean-classification-code'

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()
def merge_tree_exact(source,target):
    for item in sorted(Path(source).rglob('*')):
        if not item.is_file(): continue
        dst=Path(target)/item.relative_to(source)
        dst.parent.mkdir(parents=True,exist_ok=True)
        if dst.is_file() and sha256_file(item)!=sha256_file(dst):
            raise RuntimeError(f'Input conflict: {item.relative_to(source)}')
        if not dst.exists(): shutil.copy2(item,dst)

prior=sorted(p for p in INPUT.rglob('coffee17-preprocessing-project') if p.is_dir())
if not prior: raise FileNotFoundError('Tambahkan output notebook W0 sebagai Kaggle Input.')
for p in prior: merge_tree_exact(p,PROJECT)

if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--quiet','--no-checkout','https://github.com/ediprin/coffee-bean-classification.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--quiet','--detach',CODE_COMMIT],check=True)
lock=PROJECT/'evidence/coffee17-preprocessing-runtime-v1/requirements_preprocessing_study_lock.txt'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(lock)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from bilinear_lmmd.experiments.preprocessing_environment import verify_environment
from bilinear_lmmd.experiments.run_preprocessing_primary_confirmation import build_primary_confirmation
ENV=PROJECT/'evidence/coffee17-preprocessing-runtime-v1/runtime_environment.json'
verify_environment(ENV)
OUT=PROJECT/'experiments/coffee17-preprocessing-primary-v1'
DATA=PROJECT/'evidence/coffee17-preprocessing-data-v1'
AUTH_DIR=PROJECT/'evidence/coffee17-preprocessing-primary-v1'
AUTH_DIR.mkdir(parents=True,exist_ok=True)
AUTH=AUTH_DIR/'preprocessing_primary_confirmation.json'
result=build_primary_confirmation(
    OUT,DATA/'clean_manifest.json',DATA/'fold_manifest.json',AUTH
)
print('DECISION:',result['decision'])
print('COMPLETED RUNS:',result['completed_runs'])
print('TEST ACCESSED:',result['test_images_accessed'])
print('AUTHORITY:',AUTH)
shutil.rmtree(REPO,ignore_errors=True)
print('Klik Save Version. Output berikutnya yang dipakai adalah coffee17-preprocessing-project.')
